# AutoML · M02: LazyPredict

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M02 — LazyPredict |

---

## 🎯 Qué hace

Screening automático de ~30 clasificadores de scikit-learn con parámetros
por defecto usando LazyPredict. Objetivo: identificar qué familias de modelos
son más prometedoras antes de invertir en optimización con PyCaret, H2O
y AutoGluon.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- `data/automl/results_baselines.parquet` — resultados M01 (para comparativa)
- Entorno: `tfm_abandono`
- Librería: `lazypredict` (`pip install lazypredict`)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_lazypredict.parquet` | Métricas de ~30 modelos |
| `docs/html/fase_automl/m02_lazypredict.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ Split 70/30 estratificado (igual que M01)
    ↓ Imputación mediana
    ↓ LazyClassifier (~30 modelos, parámetros por defecto)
    ↓ Métricas extendidas: F1, AUC, Precision, Recall, MCC
    → results_lazypredict.parquet + HTML
```

## ➡️ Siguiente

`fautoml_m03_pycaret.ipynb` — AutoML con validación cruzada real

---

> **Nota metodológica:** LazyPredict NO realiza validación cruzada real.
> Sus métricas son orientativas — útiles para descartar familias de modelos,
> no para comparar directamente con los resultados de M01 (CV=5) o M03-M05.
> El split 70/30 es idéntico al de M01 para maximizar la comparabilidad.


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Verifica dependencias, detecta ROOT y carga librerías.
# LazyPredict requiere instalación separada si no está en el entorno.
# ============================================================================

import sys
import warnings
import time
import subprocess
from pathlib import Path

warnings.filterwarnings('ignore')

# Verificar e instalar lazypredict si falta
try:
    import lazypredict
except ImportError:
    print('⚠️  Instalando lazypredict...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'lazypredict'])
    print('✅ lazypredict instalado')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

# Rutas y constantes
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
TEST_SIZE     = 0.30
FRAMEWORK     = 'lazypredict'
NOTEBOOK_NAME = 'fautoml_m02_lazypredict.ipynb'
CARPETA_NB    = 'fase_automl'

info_entorno()
print('\n✅ Configuración completada')


✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================
# Dataset canónico: dataset_final_tfm.parquet (24 features + abandono).
# Split 70/30 estratificado — idéntico a M01 para comparabilidad.
# Imputación con mediana: LazyPredict no maneja NaN internamente.
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET     = 'abandono'
X          = df.drop(columns=[TARGET])
y          = df[TARGET]
n_total    = len(df)
n_features = len(X.columns)
n_abandono = int(y.sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset : {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')

# Verificación anti-leakage
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in X.columns]
if leakage_presente:
    raise ValueError(f'LEAKAGE DETECTADO: {leakage_presente}')
print('\n✅ Verificación anti-leakage superada')

# Split 70/30 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

# Imputación mediana — LazyPredict no maneja NaN
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns, index=X_test.index
)

print(f'\nSplit 70/30 estratificado:')
print(f'  Train : {fmt(len(X_train))} ({(y_train==1).mean()*100:.1f}% abandono)')
print(f'  Test  : {fmt(len(X_test))}  ({(y_test==1).mean()*100:.1f}% abandono)')
print(f'\n📋 Features ({n_features}):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO



✓ Dataset : dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)

✅ Verificación anti-leakage superada

Split 70/30 estratificado:
  Train : 23.534 (29.2% abandono)
  Test  : 10.087  (29.2% abandono)

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: EJECUTAR LAZYPREDICT
# ============================================================================
# LazyClassifier entrena ~30 clasificadores con parámetros por defecto.
# predictions=True para poder calcular métricas extendidas (MCC, Kappa)
# que LazyPredict no calcula por defecto.
# IMPORTANTE: LazyPredict no da probabilidades → log_loss no disponible.
# ============================================================================

print('\n' + '='*70)
print('EJECUTANDO LAZYPREDICT')
print('='*70)

t0 = time.time()
clf = LazyClassifier(
    verbose         = 0,
    ignore_warnings = True,
    custom_metric   = None,
    predictions     = True
)
models_df, predictions_df = clf.fit(
    X_train_imp, X_test_imp, y_train, y_test
)
t_lazy = time.time() - t0
print(f'\n✅ {len(models_df)} modelos entrenados en {t_lazy:.1f}s')

# Top 10 por F1
print(f'\n--- TOP 10 (por F1 Score) ---')
print(models_df.head(10).to_string())



EJECUTANDO LAZYPREDICT


  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 6883, number of negative: 16651
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1598
[LightGBM] [Info] Number of data points in the train set: 23534, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.292470 -> initscore=-0.883416
[LightGBM] [Info] Start training from score -0.883416



✅ 27 modelos entrenados en 158.3s

--- TOP 10 (por F1 Score) ---
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  Time Taken
Model                                                                                    
XGBClassifier                      0.90               0.87     0.87      0.90        0.82
LGBMClassifier                     0.90               0.87     0.87      0.90        0.85
RandomForestClassifier             0.90               0.86     0.86      0.90        3.38
BaggingClassifier                  0.89               0.85     0.85      0.89        1.62
ExtraTreesClassifier               0.90               0.85     0.85      0.89        2.80
SVC                                0.89               0.84     0.84      0.89       17.15
QuadraticDiscriminantAnalysis      0.83               0.83     0.83      0.84        0.07
DecisionTreeClassifier             0.85               0.83     0.83      0.85        0.21
KNeighborsClassifier              

In [4]:
# ============================================================================
# CELDA 4: CALCULAR MÉTRICAS EXTENDIDAS
# ============================================================================
# LazyPredict solo da: Accuracy, Balanced Accuracy, ROC AUC, F1, Tiempo.
# Añadimos: Precision, Recall, MCC, Kappa — para comparabilidad con M01.
# Si un modelo no tiene predicciones en predictions_df, usamos las
# métricas de LazyPredict directamente con MCC/Kappa = 0.
# ============================================================================

print('\nCalculando métricas extendidas...')
all_results = []

for model_name in models_df.index:
    row = models_df.loc[model_name]

    if model_name in predictions_df.columns:
        y_pred = predictions_df[model_name].values
        prec   = precision_score(y_test, y_pred, zero_division=0)
        rec    = recall_score(y_test, y_pred, zero_division=0)
        f1     = f1_score(y_test, y_pred, zero_division=0)
        mcc    = matthews_corrcoef(y_test, y_pred)
        kappa  = cohen_kappa_score(y_test, y_pred)
        acc    = accuracy_score(y_test, y_pred)
        bal    = balanced_accuracy_score(y_test, y_pred)
    else:
        # Sin predicciones — usar métricas de LazyPredict directamente
        prec  = 0.0
        rec   = 0.0
        f1    = float(row.get('F1 Score', 0))
        mcc   = 0.0
        kappa = 0.0
        acc   = float(row.get('Accuracy', 0))
        bal   = float(row.get('Balanced Accuracy', 0))

    all_results.append({
        'framework'        : FRAMEWORK,
        'model_name'       : model_name,
        'accuracy'         : acc,
        'balanced_accuracy': bal,
        'precision'        : prec,
        'recall'           : rec,
        'f1'               : f1,
        'auc_roc'          : float(row.get('ROC AUC', 0)),
        'mcc'              : mcc,
        'kappa'            : kappa,
        'log_loss'         : 999,   # LazyPredict no da probabilidades
        'train_time_sec'   : round(float(row.get('Time Taken', 0)), 2),
    })

df_resultados = (
    pd.DataFrame(all_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

print(f'\n--- RANKING FINAL (por F1) ---')
cols_show = ['model_name', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'train_time_sec']
print(df_resultados[cols_show].head(15).to_string(index=False))



Calculando métricas extendidas...



--- RANKING FINAL (por F1) ---
                   model_name   f1  auc_roc  precision  recall  mcc  train_time_sec
                XGBClassifier 0.83     0.87       0.87    0.79 0.76            0.82
               LGBMClassifier 0.83     0.87       0.87    0.79 0.76            0.85
       RandomForestClassifier 0.82     0.86       0.87    0.78 0.75            3.38
         ExtraTreesClassifier 0.81     0.85       0.88    0.75 0.74            2.80
            BaggingClassifier 0.80     0.85       0.86    0.75 0.73            1.62
                          SVC 0.79     0.84       0.86    0.73 0.72           17.15
         KNeighborsClassifier 0.76     0.82       0.80    0.71 0.67            3.72
       DecisionTreeClassifier 0.75     0.83       0.75    0.76 0.65            0.21
           LogisticRegression 0.74     0.81       0.79    0.70 0.65            0.09
                    LinearSVC 0.74     0.81       0.79    0.70 0.65            0.19
QuadraticDiscriminantAnalysis 0.74     0.83 

In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_lazypredict.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} modelos guardados')


💾 results_lazypredict.parquet: 27 modelos guardados


In [6]:
# ============================================================================
# CELDA 6: GRÁFICOS Y HTML
# ============================================================================
# Gráfico 1: Top 15 modelos — barras horizontales F1 + puntos AUC.
# Gráfico 2: Top 5 — barras agrupadas con todas las métricas.
# Comparativa vs M01 baselines si el parquet existe.
# render_pagina infiere título y navegación del nombre del notebook.
# ============================================================================

graficos_b64 = {}

# --- Gráfico 1: Top 15 barras horizontales ---
df_plot = df_resultados.head(15).sort_values('f1', ascending=True).copy()

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(df_plot))
bars  = ax.barh(y_pos, df_plot['f1'], color='#3182ce', alpha=0.85, height=0.6)
ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=60, zorder=5, label='AUC-ROC')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['model_name'], fontsize=9)
ax.set_xlabel('Score')
ax.set_title('LazyPredict: Top 15 Modelos (F1 + AUC)', fontweight='bold', fontsize=14)
ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, df_plot['f1']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#2d3748')
plt.tight_layout()
graficos_b64['top15'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 2: Top 5 barras agrupadas ---
df_top5    = df_resultados.head(5).copy()
metricas   = ['accuracy', 'f1', 'auc_roc', 'mcc', 'balanced_accuracy']
etiquetas  = ['Exactitud', 'F1', 'AUC-ROC', 'MCC', 'Bal. Accuracy']
colores    = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(df_top5))
width = 0.15
for j, (met, etiq, col) in enumerate(zip(metricas, etiquetas, colores)):
    ax.bar(x + j*width, df_top5[met], width, label=etiq, color=col)
ax.set_xticks(x + width*2)
ax.set_xticklabels(df_top5['model_name'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('LazyPredict: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['top5'] = figura_a_base64(fig)
plt.close()

# --- KPIs ---
mejor = df_resultados.iloc[0]
KPIS = [
    {'valor': fmt(n_total),              'titulo': 'Expedientes'},
    {'valor': str(n_features),           'titulo': 'Features'},
    {'valor': str(len(df_resultados)),   'titulo': 'Modelos evaluados'},
    {'valor': f'{mejor["f1"]:.4f}',      'titulo': f'Mejor F1 ({mejor["model_name"]})'},
]

# --- Sección metodología ---
s_metod = generar_seccion_html('Metodología', f'''
<p><strong>LazyPredict</strong> entrena automáticamente {len(df_resultados)} clasificadores
de scikit-learn con hiperparámetros por defecto sobre el dataset canónico
de producción ({fmt(n_total)} expedientes, {n_features} features).</p>
<div style="background:#fff5f5;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #e53e3e;">
  <strong>Limitación importante:</strong> LazyPredict NO realiza validación cruzada.
  Sus métricas son orientativas con parámetros por defecto.
  No son comparables directamente con los resultados de M01 (CV=5)
  ni con M03-M05 (AutoML con tuning). Su valor es identificar
  qué familias de modelos merecen optimización posterior.
</div>
<div style="background:#ebf8ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #3182ce;">
  <strong>Split:</strong> 70/30 estratificado, random_state=42
  (idéntico a M01 para maximizar comparabilidad).
  Imputación con mediana antes de LazyPredict (no maneja NaN internamente).
</div>''', 'ℹ️')

# --- Sección gráficos ---
graf_top15 = f'<img src="data:image/png;base64,{graficos_b64["top15"]}" style="max-width:100%;border-radius:8px;">'
graf_top5  = f'<img src="data:image/png;base64,{graficos_b64["top5"]}" style="max-width:100%;border-radius:8px;">'
s_graficos = generar_seccion_html('Top 15 modelos', graf_top15 + '<br>' + graf_top5, '📊')

# --- Tabla completa ---
tabla = '<table style="width:100%;border-collapse:collapse;font-size:11px;">\n'
tabla += '<tr style="background:#3182ce;color:white;">'
for col in ['#', 'Modelo', 'F1', 'AUC', 'Precision', 'Recall', 'MCC', 'Kappa', 'Tiempo']:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'

for rank, (i, row) in enumerate(df_resultados.iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str = medallas.get(rank, str(rank))
    nombre_modelo = row['model_name']
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;">{rank_str}</td>'
    tabla += f'<td style="padding:6px;">{nombre_modelo}</td>'
    for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc', 'kappa']:
        v = row[campo]
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
    tabla += f'<td style="text-align:center;">{row["train_time_sec"]:.1f}s</td>'
    tabla += '</tr>\n'
tabla += '</table>'
s_tabla = generar_seccion_html('Ranking completo de modelos', tabla, '🏆')

# --- Comparativa vs M01 baselines ---
ruta_baselines = RUTA_AUTOML / 'results_baselines.parquet'
s_comparativa = ''
if ruta_baselines.exists():
    df_base   = pd.read_parquet(ruta_baselines)
    no_dummy  = ~df_base['model_name'].str.startswith('Dummy')
    mejor_b   = df_base[no_dummy].sort_values('f1', ascending=False).iloc[0]
    mejor_lp  = df_resultados.iloc[0]

    comp = '<table style="width:100%;border-collapse:collapse;">\n'
    comp += '<tr style="background:#3182ce;color:white;">'
    for col in ['Módulo', 'Mejor Modelo', 'F1', 'AUC', 'MCC', 'CV']:
        comp += f'<th style="padding:10px;text-align:center;">{col}</th>'
    comp += '</tr>\n'

    nombre_b = mejor_b['model_name']
    comp += f'<tr style="background:#f7fafc;">'
    comp += f'<td style="padding:8px;"><strong>M01 Baselines</strong></td>'
    comp += f'<td style="padding:8px;">{nombre_b}</td>'
    comp += f'<td style="text-align:center;">{mejor_b["f1"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor_b["auc_roc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor_b["mcc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">CV=5 ✅</td></tr>\n'

    nombre_lp = mejor_lp['model_name']
    comp += f'<tr style="background:white;">'
    comp += f'<td style="padding:8px;"><strong>M02 LazyPredict</strong></td>'
    comp += f'<td style="padding:8px;">{nombre_lp}</td>'
    comp += f'<td style="text-align:center;"><strong>{mejor_lp["f1"]:.4f}</strong></td>'
    comp += f'<td style="text-align:center;"><strong>{mejor_lp["auc_roc"]:.4f}</strong></td>'
    comp += f'<td style="text-align:center;"><strong>{mejor_lp["mcc"]:.4f}</strong></td>'
    comp += f'<td style="text-align:center;color:#e53e3e;">Sin CV ⚠️</td></tr>\n'
    comp += '</table>'
    comp += '''
<div style="background:#fffaf0;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #ed8936;">
  <strong>Importante:</strong> La comparativa directa de F1 entre M01 y M02
  no es estadísticamente válida. M01 usa CV=5 (resultado estable);
  M02 usa un único split sin validación cruzada (resultado orientativo).
  El valor de M02 es la amplitud del screening: identifica familias
  prometedoras para optimizar en M03-M05.
</div>'''
    s_comparativa = generar_seccion_html('Comparativa vs M01 Baselines', comp, '🔄')
else:
    print('⚠️  results_baselines.parquet no encontrado — se omite comparativa.')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm02_lazypredict.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_metod + s_graficos + s_tabla + s_comparativa,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m02_lazypredict.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

mejor = df_resultados.iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M02 COMPLETADO')
print('='*60)
print(f'Framework     : LazyPredict')
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} x {n_features+1})')
print(f'Modelos       : {len(df_resultados)}')
print(f'Mejor modelo  : {mejor["model_name"]}')
print(f'  F1 test     : {mejor["f1"]:.4f}  (sin CV — orientativo)')
print(f'  AUC         : {mejor["auc_roc"]:.4f}')
print(f'  MCC         : {mejor["mcc"]:.4f}')
print(f'Resultados    : {RUTA_AUTOML / "results_lazypredict.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m02_lazypredict.html"}')
print(f'\n➡️  Siguiente  : fautoml_m03_pycaret.ipynb')
print('='*60)



✅ AUTOML-M02 COMPLETADO
Framework     : LazyPredict
Dataset       : dataset_final_tfm.parquet (33.621 x 25)
Modelos       : 27
Mejor modelo  : XGBClassifier
  F1 test     : 0.8296  (sin CV — orientativo)
  AUC         : 0.8724
  MCC         : 0.7649
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_lazypredict.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m02_lazypredict.html

➡️  Siguiente  : fautoml_m03_pycaret.ipynb
